<a href="https://colab.research.google.com/github/iambekzodboboev/traffic-sign-recognition/blob/master/notebooks/01_dataset_download_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset download (Colab, from Google Drive)

The dataset (~14.9GB, 200 classes) is already uploaded to the user's own
Google Drive. We download it directly into the Colab session's local disk
using Colab's authenticated Google auth + the Drive API (downloading by
file ID, as the file's own owner), rather than mounting Drive and training
against it directly — the dataset has 116,642 individual small image
files, and reading that many small files over a mounted Drive during
training would be slow. A single large sequential download, then local
unzip, avoids that.

**Why not `gdown` / a public share link:** we initially used `gdown` with
a public "anyone with the link" share URL, but Google Drive rate-limits
public link downloads after repeated use ("Too many users have viewed or
downloaded this file recently"), which we hit during testing. Downloading
via the authenticated Drive API instead avoids that quota entirely, since
it's not going through the public-link path.

**Every Colab session:** run the cells below in order. You'll get a
one-time login/consent popup for the Drive API access.

In [ ]:
from google.colab import auth
auth.authenticate_user()

import io
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

FILE_ID = "1Mi0IleRucNmwnQ4g_ZEWOyBFFv4mO9ba"
ZIP_PATH = "/content/traffic_sign_dataset.zip"

drive_service = build('drive', 'v3')
request = drive_service.files().get_media(fileId=FILE_ID)

with io.FileIO(ZIP_PATH, 'wb') as fh:
    downloader = MediaIoBaseDownload(fh, request, chunksize=200 * 1024 * 1024)
    done = False
    while not done:
        status, done = downloader.next_chunk()
        print(f"Download progress: {int(status.progress() * 100)}%")

print("Download complete.")

In [4]:
import os

size_bytes = os.path.getsize(ZIP_PATH)
print(f"Downloaded file size: {size_bytes:,} bytes ({size_bytes / 1e9:.2f} GB)")
print("Expected (from local audit): 14,945,416,206 bytes (~14.9 GB)")

Downloaded file size: 14,945,416,206 bytes (14.95 GB)
Expected (from local audit): 14,945,416,206 bytes (~14.9 GB)


In [ ]:
!rm -rf /content/dataset
!mkdir -p /content/dataset
!unzip -q "$ZIP_PATH" -d /content/dataset
print("Unzip done.")

In [6]:
# Quick sanity check: should match the local audit (200 classes under Data/)
data_dir = '/content/dataset/Data'
classes = sorted(os.listdir(data_dir), key=int)
print('Number of classes:', len(classes))
print('First 5 classes:', classes[:5])
print('Last 5 classes:', classes[-5:])

total_images = sum(len(os.listdir(os.path.join(data_dir, c))) for c in classes)
print('Total images:', total_images)

Number of classes: 200
First 5 classes: ['0', '1', '2', '3', '4']
Last 5 classes: ['195', '196', '197', '198', '199']
Total images: 116642


If this prints 200 classes and 116,642 total images, the download matches what we audited locally, and we're ready to move on to the data audit / EDA notebook (roadmap stage 3).